# Módulo 1 — Introducción a Apache Spark
### Big Data DD283 | Universidad Autónoma del Perú

**Estudiante**: Jhonn Viequier Lindo Barrientos

> **Nota:** Este notebook funciona en **Databricks** (donde `spark` ya existe) y en **Google Colab** (donde hay que crear la sesión). La primera celda detecta el entorno automáticamente. Las rutas de los CSV se ajustan según dónde se ejecute.

### Configuración del entorno (Databricks o Colab)

In [ ]:
# Si estas en COLAB, descomenta la siguiente linea:
# !pip install pyspark -q

# Detectar entorno: en Databricks 'spark' ya existe; en Colab hay que crearlo
try:
    spark
    print("Entorno Databricks: Spark ya disponible.")
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Modulo1Spark").master("local[2]").getOrCreate()
    print("Entorno Colab: sesion Spark creada.")


### Verificación de versión de Spark

In [ ]:
#VERIFICACION DE VERSION DE SPARK
print("Spark version:", spark.version)

### Creación de un DataFrame pequeño

In [ ]:
#CREACION DE UN DATAFRAME PEQUEÑO
data = [("Juan", 25), ("María", 30), ("Pedro", 35)]
df = spark.createDataFrame(data, ["nombre", "edad"])
df.show()

### Transformaciones y acciones

In [ ]:
# Transformación corta
df2 = df.filter(df.edad > 28)   # transformación
# Retorno de resultados
print("Filtrados:", df2.count())
df2.show()

### Cargar un CSV en Spark — productos.csv

El archivo `productos.csv` debe estar disponible. En Databricks va en `/Volumes/...`; en Colab se sube y se lee con ruta local.

In [ ]:
# En DATABRICKS (ruta original del profesor):
# df_productos = spark.read.csv('/Volumes/workspace/default/resources/productos.csv', header=True, inferSchema=True)
# display(df_productos)

# En COLAB / local: sube productos.csv y usa la ruta local
ruta_productos = "productos.csv"   # ajusta la ruta si lo subes a otra carpeta

df_productos = spark.read.csv(ruta_productos, header=True, inferSchema=True)

# show() funciona en ambos entornos (display() es solo de Databricks)
df_productos.show()
print("Esquema detectado:")
df_productos.printSchema()
print(f"Total de productos: {df_productos.count()}")

## Mini-ejercicio

**Crear un .csv de Datos Personales y mostrar los 10 primeros registros.**

Resolución en 2 pasos: (1) crear el archivo `datos_personales.csv`, (2) cargarlo en Spark y mostrar los 10 primeros registros.

In [ ]:
# ============================================================
# PASO 1: Crear el archivo datos_personales.csv
# ============================================================
import csv

datos = [
    ["id", "nombre", "apellido", "edad", "ciudad", "profesion", "salario"],
    [1, "Juan", "Quispe", 28, "Lima", "Ingeniero", 4500],
    [2, "Maria", "Torres", 34, "Arequipa", "Doctora", 7200],
    [3, "Pedro", "Mamani", 41, "Cusco", "Abogado", 5800],
    [4, "Lucia", "Flores", 25, "Trujillo", "Disenadora", 3200],
    [5, "Carlos", "Ramos", 38, "Piura", "Contador", 4100],
    [6, "Ana", "Vargas", 29, "Lima", "Arquitecta", 5500],
    [7, "Jose", "Huaman", 45, "Junin", "Profesor", 3800],
    [8, "Rosa", "Chavez", 31, "Lambayeque", "Enfermera", 3500],
    [9, "Miguel", "Rojas", 52, "Ancash", "Gerente", 9500],
    [10, "Carmen", "Diaz", 27, "Lima", "Programadora", 4800],
    [11, "Luis", "Castro", 36, "Loreto", "Biologo", 4200],
    [12, "Elena", "Morales", 43, "Tacna", "Psicologa", 5100],
    [13, "Jorge", "Paredes", 30, "Ica", "Veterinario", 3900],
    [14, "Sofia", "Reyes", 26, "Cajamarca", "Periodista", 3300],
    [15, "Raul", "Mendoza", 48, "Puno", "Economista", 6700],
]

with open("datos_personales.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(datos)

print("Archivo datos_personales.csv creado con", len(datos)-1, "registros")

In [ ]:
# ============================================================
# PASO 2: Cargar el CSV en Spark y mostrar los 10 primeros registros
# ============================================================
df_personas = spark.read.csv("datos_personales.csv", header=True, inferSchema=True)

print("Esquema detectado:")
df_personas.printSchema()

print("\nPrimeros 10 registros:")
df_personas.show(10)

# Tambien se puede obtener como lista con take(10) o limit(10)
print("Total de registros en el archivo:", df_personas.count())

### Extra — pequeño análisis sobre los datos personales

Para demostrar transformaciones de Spark sobre el CSV creado.

In [ ]:
from pyspark.sql import functions as F

# Salario promedio por ciudad (groupBy + avg)
print("Salario promedio por ciudad:")
df_personas.groupBy("ciudad") \
    .agg(F.round(F.avg("salario"), 0).alias("salario_promedio"),
         F.count("*").alias("personas")) \
    .orderBy("salario_promedio", ascending=False) \
    .show()

# Personas mayores de 40 anios
print("Personas mayores de 40 anios:")
df_personas.filter(df_personas.edad > 40).select("nombre", "apellido", "edad", "profesion").show()